# ⚙️ Technical Architecture
## Investment MCP Multi-Agent System

---

> **Purpose:** An engineering-focused breakdown of the full system architecture — for technical reviewers, interviewers, and portfolio evaluators.

---

## 1. Architecture Overview

### Layer Diagram

```
┌────────────────────────────────────────────────────────┐
│  PRESENTATION LAYER                                    │
│  Streamlit UI (port 8501)                              │
│  Pages: Home · Analyze · Results · History             │
└───────────────────────┬────────────────────────────────┘
                        │ REST (httpx / requests)
┌───────────────────────▼────────────────────────────────┐
│  API LAYER                                             │
│  FastAPI (port 8000)                                   │
│  Async · Pydantic v2 · CorrelationID middleware        │
│  Auth: X-API-Key  Rate: slowapi 10/min/IP              │
└───────────────────────┬────────────────────────────────┘
                        │ ThreadPoolExecutor (max 5)
┌───────────────────────▼────────────────────────────────┐
│  ORCHESTRATION LAYER                                   │
│  CrewAI (sequential Process)                           │
│  5 agents · 5 tasks · context chaining                │
│  LLM: Anthropic Claude (claude-sonnet-4-6)            │
└───────────────────────┬────────────────────────────────┘
                        │ MCPGateway.call(tool_name, payload)
┌───────────────────────▼────────────────────────────────┐
│  MCP TOOL GATEWAY                                      │
│  MCPRegistry → MCPBaseTool → validate_input → run()   │
│  6 tools, each backed by a service                    │
└───────────┬───────────────────────────────────────────┘
            │
┌───────────▼──────────────────────────────────────────┐
│  SERVICES LAYER                                      │
│  MarketData · Financials · Risk · News · LLM · Report│
└─────────���─┬──────────────────────────────────────────┘
            │
┌───────────▼──────────────────────────────────────────┐
│  DATA LAYER                                          │
│  PostgreSQL 16 + pgvector                            │
│  Async (asyncpg) + Sync (psycopg2)                   │
│  Tables: tickers · analysis_runs · agent_outputs     │
│          reports                                     │
└──────────────────────────────────────────────────────┘
```

---

## 2. MCP Tool Gateway — Design Deep Dive

### What is MCP?

**MCP (Model Context Protocol)** is an open protocol originally from Anthropic that standardizes how AI models communicate with external tools. In this system, we implement a **gateway pattern inspired by MCP** to decouple agent logic from data service implementations.

### Why This Matters

Without a gateway pattern, agents would call services directly — coupling LLM tool descriptions to service internals and making validation ad-hoc.

With the MCP Gateway:
- Every tool has a **declared schema** (Pydantic model)
- Every call is **validated before execution**
- The gateway is a **thread-safe singleton** (double-checked locking)
- Agents don't know which service backs a tool — they call by name

### Class Hierarchy

```python
class MCPBaseTool(ABC):
    name: str                        # e.g. "get_stock_price"
    description: str
    input_schema: Type[BaseModel]    # Pydantic — validated before run()

    def execute(self, payload: dict) -> MCPToolResult:
        inputs = self.validate_input(payload)  # raises ToolExecutionError on bad input
        return self.run(inputs)

    @abstractmethod
    def run(self, inputs: BaseModel) -> dict[str, Any]: ...


class StockPriceTool(MCPBaseTool):
    name = "get_stock_price"
    input_schema = StockPriceInput   # {ticker, period, interval}

    def run(self, inputs):
        return market_data_service.get_price_history(inputs.ticker, inputs.period)
```

### Thread Safety

The gateway singleton uses **double-checked locking** to be safe across concurrent requests:

```python
_gateway_lock = threading.Lock()

def get_gateway() -> MCPGateway:
    global _gateway_instance
    if _gateway_instance is None:
        with _gateway_lock:
            if _gateway_instance is None:  # second check inside lock
                _gateway_instance = create_gateway()
    return _gateway_instance
```

---

## 3. The 6 MCP Tools

| Tool | Input Schema | Service Backed By | Output |
|------|-------------|------------------|--------|
| `get_stock_price` | `ticker, period, interval` | `MarketDataService` (yfinance) | Price history, OHLCV, current price, % change |
| `get_financial_data` | `ticker, statement_type?` | `FinancialsService` (yfinance) | EPS, P/E, margins, revenue, debt, key metrics |
| `calculate_risk_metrics` | `ticker, period` | `RiskService` (yfinance + numpy) | Beta, volatility, drawdown, Sharpe, VaR, RSI, MACD |
| `get_news_sentiment` | `ticker, days?` | `NewsService` (NewsAPI) | Headlines, sentiment score, label |
| `get_sector_analysis` | `ticker` | `MarketDataService` (yfinance) | Sector ETF comparison, relative performance |
| `save_report` | `run_id, ticker, content, structured?` | `ReportService` (PostgreSQL) | Validates sections, upserts to DB, marks run COMPLETED |

---

## 4. Async FastAPI + Sync CrewAI Integration

### The Problem

FastAPI is async-first. CrewAI's `crew.kickoff()` is synchronous and can run for 60–120 seconds. Running it directly in an async route would block the event loop.

### The Solution

```python
# analysis.py — route handler
_executor = ThreadPoolExecutor(max_workers=5)

@router.post("/analyze")
async def create_analysis(body: AnalysisRequest, background_tasks: BackgroundTasks):
    run = await repo.create(ticker=body.ticker)
    background_tasks.add_task(_run_crew_background, run.id, run.ticker, body.period)
    return AnalysisResponse(run_id=run.id, status="PENDING")

async def _run_crew_background(run_id, ticker, period):
    run_id_var.set(run_id)              # set ContextVar in async context
    ctx = copy_context()                # snapshot ContextVars
    await loop.run_in_executor(         # run crew in thread pool
        _executor, ctx.run, _run_crew_sync, run_id, ticker, period
    )  # ctx.run propagates ContextVars into the thread
```

### Why `copy_context()` Matters

`ContextVar` values are task-local in Python's async model. Without `copy_context()`, the `run_id_var` set in the async task would be invisible inside the `ThreadPoolExecutor` thread — breaking structured logging and correlation tracking across the full crew execution.

---

## 5. Database Design

### Schema (4 Tables)

```sql
tickers
  id UUID PK | symbol VARCHAR(10) UNIQUE | company_name | sector | exchange

analysis_runs
  id UUID PK | ticker VARCHAR(10) | ticker_id FK(tickers) |
  status ENUM(PENDING,RUNNING,COMPLETED,FAILED) | config JSONB |
  created_at | started_at | completed_at | error_message

agent_outputs
  id UUID PK | run_id FK(analysis_runs) CASCADE | agent_name |
  output_data JSONB | tool_calls JSONB | created_at

reports
  id UUID PK | run_id FK(analysis_runs) CASCADE UNIQUE |
  ticker_symbol | content TEXT | structured JSONB | created_at
```

### Dual Session Strategy

```
FastAPI routes  →  AsyncSession (asyncpg driver)
CrewAI tools    →  SyncSession  (psycopg2 driver)
```

Mixing sync and async SQLAlchemy sessions in the same process is safe because they use different connection pools backed by different drivers. The MCP tools run in executor threads where `await` is not available.

### Migration Strategy

- **Local dev (default):** `SQLAlchemy Base.metadata.create_all()` at startup — safe for fresh databases
- **Production:** `USE_ALEMBIC=true` → runs `alembic upgrade head` — proper incremental migrations with `downgrade()` support
- Initial migration: `alembic/versions/20260419_0001_initial_schema.py`

---

## 6. Risk Calculation Services

All calculations are performed from raw daily price data fetched via yfinance:

```python
# RSI (14-period, Wilder smoothing)
delta = prices.diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
# Guard: loss=0 → RSI=100 if any gains, else 50 (prevents NaN/∞)

# Annualised Volatility
returns = prices.pct_change().dropna()
volatility = returns.std() * (252 ** 0.5)

# Sharpe Ratio (risk-free rate = 4.5% annualised)
excess_returns = returns - (0.045 / 252)
sharpe = (excess_returns.mean() / excess_returns.std()) * (252 ** 0.5)

# Max Drawdown
cumulative = (1 + returns).cumprod()
rolling_max = cumulative.cummax()
drawdown = (cumulative - rolling_max) / rolling_max
max_drawdown = drawdown.min()

# VaR 95% (historical simulation)
var_95 = returns.quantile(0.05)
```

---

## 7. Testing Strategy

### Four Layers

| Layer | File Count | Test Count | Isolation |
|-------|-----------|-----------|----------|
| **Unit** | 7 files | ~51 tests | Pure Python, no I/O — mocks for all external deps |
| **Integration** | 4 files | ~39 tests | FastAPI TestClient + in-memory SQLite (StaticPool) |
| **Smoke** | 2 files | ~17 tests | Module import checks, Alembic setup, tool registration |
| **E2E** | 1 file | 5 tests | Full simulated crew run via API flow |

### Key Test Isolation Design

```python
# conftest.py — in-memory SQLite for tests
TEST_ASYNC_URL = "sqlite+aiosqlite:///:memory:"
engine = create_async_engine(TEST_ASYNC_URL, poolclass=StaticPool)

# FastAPI dependency override
test_app.dependency_overrides[get_async_session] = lambda: async_db_session
test_app.dependency_overrides[get_gateway] = lambda: mock_gateway
```

This means integration tests run the real FastAPI application with real route logic, but all I/O (DB, gateway, services) is injected via overrides — no external dependencies, no network calls.

---

## 8. Deployment Architecture

### Docker Compose (Local)

```yaml
services:
  postgres:   pgvector/pgvector:pg16  (port 5432)
  backend:    FastAPI + CrewAI        (port 8000)  depends_on: postgres healthy
  ui:         Streamlit               (port 8501)  depends_on: backend healthy
  prometheus: prom/prometheus         (port 9090)
  grafana:    grafana/grafana         (port 3000)
```

### Kubernetes (Production)

```
Namespace: investment-system
├── backend-deployment    (2 replicas, liveness + readiness probes)
├── ui-deployment         (1 replica, LoadBalancer service)
├── postgres-statefulset  (1 replica, 20Gi PVC)
├── configmap             (non-secret env vars)
└── secret                (ANTHROPIC_API_KEY, DB credentials)
```

### Cloud (Terraform / GCP)

```
GKE Autopilot cluster  →  backend + ui pods
Cloud SQL Postgres 16  →  managed database
Artifact Registry      →  container images
```

---

## 9. Observability Stack

### Metrics (Prometheus + Grafana)

The backend exposes `/metrics` via `prometheus-fastapi-instrumentator`:
- `http_requests_total{method, path, status}` — request counter
- `http_request_duration_seconds{method, path}` — latency histogram
- Standard Python process metrics

Prometheus scrapes `backend:8000/metrics` every 15 seconds. Grafana dashboard is provisioned automatically from `monitoring/grafana/dashboards/overview.json`.

### Structured Logging

All log lines are JSON (configurable via `LOG_FORMAT=json`):

```json
{
  "timestamp": "2026-04-19T12:00:01Z",
  "level": "INFO",
  "logger": "app.crews.investment_crew",
  "message": "Crew completed",
  "run_id": "550e8400-...",
  "request_id": "abc123"
}
```

`run_id` is injected via `ContextVar` and propagated into the executor thread via `copy_context()` — so every log line throughout the full crew execution carries the same `run_id`.

---

## 10. Security Implementation

| Feature | Implementation |
|---------|---------------|
| API auth | `X-API-Key` header, `fastapi.security.APIKeyHeader`, configurable via `API_KEY` env var |
| Rate limiting | `slowapi` 10/min per IP on `POST /analyze`, configurable via `RATE_LIMIT_ANALYZE` |
| Input validation | Pydantic v2 models on all endpoints — ticker uppercasing, period whitelist |
| DB query safety | SQLAlchemy ORM only — no raw string interpolation, no SQL injection surface |
| Secrets | All credentials in environment variables, never in source code |

---

## 11. Roadmap / Future Extensibility

The architecture is designed to extend cleanly:

| Extension | How |
|-----------|-----|
| **RAG over historical reports** | `document_embeddings` table + pgvector is already provisioned. Add an embedding pipeline to ingest completed reports. |
| **Additional agents** | Create a new `MCPBaseTool` subclass, register in `MCPRegistry`, add an agent builder. No changes to existing agents. |
| **Multi-user auth** | Replace `require_api_key` dependency with JWT validator. Router wiring unchanged. |
| **Async task queue** | Replace `ThreadPoolExecutor` with Celery + Redis. The `BackgroundTasks` call site becomes a Celery task dispatch. |
| **Streaming report generation** | Add SSE endpoint that yields token-by-token output from the Report Writer using Anthropic streaming API. |
| **Alternative LLMs** | `LLMService.get_crewai_llm()` wraps one line — swap to any LiteLLM-compatible model. |

---

*See `01_demo_walkthrough.ipynb` for a user-facing demo.*  
*See `03_demo_scenarios.ipynb` for concrete stock analysis examples.*